# TA Session Analyzer — run in Colab

Runs the Streamlit app from `MAHI-0828/ta-session-analyzer` inside Colab and exposes it through a public tunnel URL so you can use it in your browser.

You'll need two free API keys:
- **Gemini** (analysis/scoring): https://aistudio.google.com/app/apikey
- **Deepgram** (transcription/diarization, free credits): https://console.deepgram.com

Run the cells top to bottom. The keys are entered with a hidden prompt (`getpass`) — they are never written to disk or shown in the notebook output.

In [ ]:
#@title 1. Clone the repo
BRANCH = "claude/ta-session-analyzer-z2n1ml"  #@param {type:"string"}

import os
if os.path.exists("/content/ta-session-analyzer"):
    %cd /content/ta-session-analyzer
    !git fetch origin $BRANCH
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    %cd /content
    !git clone -b $BRANCH https://github.com/MAHI-0828/ta-session-analyzer.git
    %cd ta-session-analyzer

In [ ]:
#@title 2. Install system + Python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements.txt
!pip install -q streamlit

In [ ]:
#@title 3. Enter your API keys (hidden input, not saved to disk)
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
os.environ["DEEPGRAM_API_KEY"] = getpass("Deepgram API key: ")
print("Keys set for this session.")

In [ ]:
#@title 4. Launch the Streamlit app + public tunnel
#@markdown Click the printed `https://*.loca.lt` link once it appears below.
#@markdown If it asks for a "Tunnel Password", use the IP address printed
#@markdown right before the link (that's normal for localtunnel — it's not a real password).

import subprocess, time, urllib.request

# Kill any previous run
!pkill -f streamlit || true
!pkill -f lt || true

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "ta_app.py", "--server.port", "8501",
     "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(8)

print("Your tunnel password/IP (if localtunnel asks for one):")
!curl -s https://loca.lt/mytunnelpassword; echo

!npx --yes localtunnel --port 8501

---
### Alternative: run the batch CLI directly (no UI, no tunnel)

If you'd rather run the CSV batch pipeline straight in the notebook instead of the Streamlit UI, use this instead of cell 4. Upload your `ta_sessions_today.csv` to `/content/ta-session-analyzer/` first (use the Colab file browser on the left, or `files.upload()`).

In [ ]:
#@title (Optional) Run the daily batch CLI instead
!python ta_session_analyzer.py ta_sessions_today.csv